session 1 - assignment-1

In [ ]:
import jwt

def issue_token(user_id, role, secret):
    return jwt.encode({"sub": user_id, "role": role}, secret, algorithm="HS256")

def read_claims(token, secret):
    return jwt.decode(token, secret, algorithms=["HS256"])

def token_is_valid(token, secret):
    try:
        jwt.decode(token, secret, algorithms=["HS256"])
        return True
    except Exception:
        return False

session 1 - assignment 2

In [ ]:
import jwt
from jwt.exceptions import ExpiredSignatureError, InvalidSignatureError, InvalidTokenError

def issue_token(user_id, role, secret, exp_ts=None):
    payload = {"sub": user_id, "role": role}
    if exp_ts is not None:
        payload["exp"] = exp_ts
    return jwt.encode(payload, secret, algorithm="HS256")

def verify_token(token, secret):
    try:
        return jwt.decode(token, secret, algorithms=["HS256"])
    except ExpiredSignatureError:
        return "expired"
    except InvalidSignatureError:
        return "invalid_signature"
    except InvalidTokenError:
        return "invalid_token"
    except Exception:
        return "invalid_token"

def extract_role(token, secret):
    result = verify_token(token, secret)
    if isinstance(result, dict):
        return result.get("role", "invalid")
    return "invalid"

session 2 - assignment - 1

In [ ]:
def count_by_status(tasks):
    counts = {}
    for task in tasks:
        status = task["status"]
        counts[status] = counts.get(status, 0) + 1
    return counts

session 2 - assignment - 2

In [ ]:
def summarize_results(cases):
    failures = []
    passed = 0

    for case in cases:
        if case["expected"] == case["actual"]:
            passed += 1
        else:
            failures.append(case["name"])

    total = len(cases)

    return {
        "total": total,
        "passed": passed,
        "failed": total - passed,
        "failures": failures
    }

session 3 - assignment - 1

In [ ]:
class TaskService:
    def __init__(self, repo, clock):
        self.repo = repo
        self.clock = clock

    def overdue_count(self):
        today = self.clock()
        count = 0

        for task in self.repo():
            due = task.get("due")
            if due is not None and task.get("status") != "done" and due < today:
                count += 1

        return count

session 3 - assignment - 2

In [ ]:
class TaskService:
    def __init__(self, repo, cache, clock):
        self.repo = repo
        self.cache = cache
        self.clock = clock

    def get_task(self, task_id):
        if task_id in self.cache:
            return self.cache[task_id]

        task = self.repo(task_id)
        if task is not None:
            self.cache[task_id] = task
        return task

    def status_of(self, task_id):
        task = self.get_task(task_id)
        if task is None:
            return "unknown"
        return task.get("status", "unknown")

    def summary(self, ids):
        counts = {}
        for task_id in ids:
            task = self.get_task(task_id)
            if task is not None:
                status = task.get("status", "unknown")
                counts[status] = counts.get(status, 0) + 1
        return counts

    def stamp(self, task_id):
        task = self.get_task(task_id)
        if task is None:
            return None
        return {**task, "checked_at": self.clock()}